In [1]:
import pyodbc
from sqlalchemy import create_engine
import urllib
import pandas as pd
import numpy as np
import os
import joblib

# Connect to SQL Server
driver = "ODBC Driver 17 for SQL Server"
server_name = r"LAPTOP-5CHV2K6U\UMASANKAR"
database_name = "ShopSenseDB"

conn_str = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server_name};"
    f"DATABASE={database_name};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(conn_str)}")

# Pull order items with categories and prices
query = """
SELECT 
    v.order_id,
    v.customer_unique_id,
    v.product_id,
    v.product_category_name_english AS category,
    v.price,
    ISNULL(r.review_score, 4) AS review_score
FROM dbo.vw_order_details v
LEFT JOIN dbo.order_reviews r 
    ON v.order_id = r.order_id
WHERE v.order_status = 'delivered'
  AND v.product_category_name_english <> 'uncategorized';
"""

with engine.connect() as conn:
    df_items = pd.read_sql(query, conn)

print(f"✓ Pulled {len(df_items):,} product line items across {df_items['product_id'].nunique():,} unique products!")
display(df_items.head(3))

✓ Pulled 109,273 product line items across 31,621 unique products!


,order_id,customer_unique_id,product_id,category,price,review_score
0,2d13edfc9500d7a00b30da501d70359b,ef8d690552f3271b4a2645e1d272abcf,37bd6327d499dd991f2da7681625bb3c,furniture_decor,19.50,4
1,2d13edfc9500d7a00b30da501d70359b,ef8d690552f3271b4a2645e1d272abcf,37bd6327d499dd991f2da7681625bb3c,furniture_decor,19.50,4
2,7f17ed9fe6d0823eee3396f86c6dea63,aba4cbce5642248fc726ee1268ccaba0,564d0015be20622253aee5b6e875bea8,telephony,29.99,3


In [2]:
# 1. Compute Category Best-Sellers (ranked by review score and sales volume)
category_top_products = df_items.groupby(["category", "product_id"]).agg(
    order_count=("order_id", "count"),
    avg_price=("price", "mean"),
    avg_rating=("review_score", "mean")
).reset_index()

# Filter for products with at least 3 purchases and rating >= 4.0
popular_products = category_top_products[
    (category_top_products["order_count"] >= 3) & 
    (category_top_products["avg_rating"] >= 4.0)
].sort_values(by=["category", "order_count", "avg_rating"], ascending=[True, False, False])

print(f"✓ Identified {len(popular_products):,} verified high-rating products across categories!")
display(popular_products.head(5))

✓ Identified 5,813 verified high-rating products across categories!


,category,product_id,order_count,avg_price,avg_rating
7,agro_industry_and_commerce,11250b0d4b709fee92441c5f34122aed,22,414.136364,4.727273
23,agro_industry_and_commerce,423a6644f0aa529e8828ff1f91003690,18,446.833333,4.333333
33,agro_industry_and_commerce,672e757f331900b9deea127a2a7b79fd,17,405.000000,4.000000
45,agro_industry_and_commerce,b5aebb467d9a92162173cbd234e00d99,10,35.000000,4.800000
37,agro_industry_and_commerce,980ecbcc15fe174ec1e5757c4d75b1bf,8,24.687500,4.625000


In [3]:
# 1. Find orders that contained more than 1 item
multi_item_orders = df_items.groupby("order_id").filter(lambda x: len(x) > 1)
print(f"Orders with multiple items: {multi_item_orders['order_id'].nunique():,}")

# 2. Self-merge to pair products purchased in the same order
order_pairs = pd.merge(
    multi_item_orders[["order_id", "product_id", "category"]],
    multi_item_orders[["order_id", "product_id", "category"]],
    on="order_id",
    suffixes=("_A", "_B")
)

# 3. Exclude pairing a product with itself
order_pairs = order_pairs[order_pairs["product_id_A"] != order_pairs["product_id_B"]]

# 4. Count co-occurrences
co_occurrences = order_pairs.groupby(["product_id_A", "product_id_B"]).size().reset_index(name="pair_count")

# Sort by highest frequency of being bought together
co_occurrences = co_occurrences.sort_values(by="pair_count", ascending=False)
print(f"✓ Mapped {len(co_occurrences):,} product affinity pairs!")
display(co_occurrences.head(5))

Orders with multiple items: 9,933
✓ Mapped 7,882 product affinity pairs!


,product_id_A,product_id_B,pair_count
1234,270516a3f41dc035aa87d220228f844c,05b515fdc76e888aada3c6d66c201dff,100
171,05b515fdc76e888aada3c6d66c201dff,270516a3f41dc035aa87d220228f844c,100
7080,e53e557d5a159f5aa2c5e995dfdf244b,36f60d45225e60c7da4558b070ce4b60,48
1723,36f60d45225e60c7da4558b070ce4b60,e53e557d5a159f5aa2c5e995dfdf244b,48
5278,a9d9db064d4afd4458eb3e139fe29167,710b7c26b7a742f497bba45fab91a25f,36


In [4]:
def recommend_for_customer(customer_id, top_n=3):
    # 1. Get products previously purchased by this customer
    customer_history = df_items[df_items["customer_unique_id"] == customer_id]
    
    if customer_history.empty:
        # Unknown / Brand New Customer -> Global Popular Products
        top_global = popular_products.head(top_n)
        return [
            {
                "product_id": row["product_id"],
                "category": row["category"],
                "score_pct": 85,
                "reason": "Top-selling platform favorite"
            }
            for _, row in top_global.iterrows()
        ]
    
    past_products = set(customer_history["product_id"].unique())
    last_bought_category = customer_history["category"].iloc[-1]
    
    recommendations = []
    
    # 2. Check Co-Occurrence (Frequently Bought Together)
    for prod in past_products:
        paired = co_occurrences[co_occurrences["product_id_A"] == prod]
        for _, row in paired.iterrows():
            target_prod = row["product_id_B"]
            if target_prod not in past_products and target_prod not in [r["product_id"] for r in recommendations]:
                recommendations.append({
                    "product_id": target_prod,
                    "category": last_bought_category,
                    "score_pct": min(95, 75 + int(row["pair_count"] * 5)),
                    "reason": f"Frequently bought together with your previous order"
                })
            if len(recommendations) >= top_n:
                break
                
    # 3. If fewer than top_n, fill with Category Best-Sellers
    if len(recommendations) < top_n:
        cat_favorites = popular_products[popular_products["category"] == last_bought_category]
        for _, row in cat_favorites.iterrows():
            target_prod = row["product_id"]
            if target_prod not in past_products and target_prod not in [r["product_id"] for r in recommendations]:
                recommendations.append({
                    "product_id": target_prod,
                    "category": row["category"],
                    "score_pct": 82,
                    "reason": f"Top-rated best-seller in your favorite category: {row['category']}"
                })
            if len(recommendations) >= top_n:
                break
                
    return recommendations[:top_n]

# Test on a real customer from our dataset
sample_cust = df_items["customer_unique_id"].iloc[50]
print(f"Testing recommendations for Customer: [{sample_cust}]")
recs = recommend_for_customer(sample_cust, top_n=3)

for idx, r in enumerate(recs, 1):
    print(f"\nRecommendation #{idx}:")
    print(f"  - Product ID:   {r['product_id']}")
    print(f"  - Category:     {r['category']}")
    print(f"  - Match Score:  {r['score_pct']}%")
    print(f"  - Explanation:  {r['reason']}")

Testing recommendations for Customer: [afc789e24a84241112316aaaba8664f8]

Recommendation #1:
  - Product ID:   c6dd917a0be2a704582055949915ab32
  - Category:     cool_stuff
  - Match Score:  82%
  - Explanation:  Top-rated best-seller in your favorite category: cool_stuff

Recommendation #2:
  - Product ID:   601a360bd2a916ecef0e88de72a6531a
  - Category:     cool_stuff
  - Match Score:  82%
  - Explanation:  Top-rated best-seller in your favorite category: cool_stuff

Recommendation #3:
  - Product ID:   54d9ac713e253fa1fae9c8003b011c2a
  - Category:     cool_stuff
  - Match Score:  82%
  - Explanation:  Top-rated best-seller in your favorite category: cool_stuff


In [5]:
MODELS_DIR = os.path.join("..", "models")

rec_data = {
    "co_occurrences": co_occurrences.head(10000), # Top 10k frequent pairs
    "popular_products": popular_products,
    "product_categories": df_items[["product_id", "category"]].drop_duplicates()
}

rec_filepath = os.path.join(MODELS_DIR, "recommendation_engine.pkl")
joblib.dump(rec_data, rec_filepath)

print(f"✓ Recommendation Engine saved to: [{rec_filepath}]!")

✓ Recommendation Engine saved to: [..\models\recommendation_engine.pkl]!
